In [9]:
import pyomo.environ as pyo
import pyomo
import os

In [10]:
# Pyomo modélise des problèmes d'optimisation linéaires et non linéaires : https://www.pyomo.org/
print(pyomo.version.version)

6.10.1


In [11]:

model = pyo.ConcreteModel()

# 4 variables continues bornées entre 1 et 5, avec un point de départ
model.x1 = pyo.Var(bounds=(1, 5), initialize=1)
model.x2 = pyo.Var(bounds=(1, 5), initialize=5)
model.x3 = pyo.Var(bounds=(1, 5), initialize=5)
model.x4 = pyo.Var(bounds=(1, 5), initialize=1)

# Objectif non linéaire
model.obj = pyo.Objective(
    expr=model.x1 * model.x4 * (model.x1 + model.x2 + model.x3) + model.x3,
    sense=pyo.minimize,
)

# Contrainte d'inégalité non linéaire : produit >= 25
model.c1 = pyo.Constraint(expr=model.x1 * model.x2 * model.x3 * model.x4 >= 25)

# Contrainte d'égalité non linéaire : somme des carrés = 40
model.c2 = pyo.Constraint(
    expr=model.x1**2 + model.x2**2 + model.x3**2 + model.x4**2 == 40
)



# contournement bug paquet ubuntu
solver = pyo.SolverFactory("ipopt", executable=os.path.expanduser("~/ipopt-nodisplay"))
solver._version_timeout = 30
#solver._version = (3, 14, 19, 0)      # évite l'appel « ipopt -v »
solver.options["max_cpu_time"] = 10   # limite de temps d'Ipopt, en secondes
# ipopt est un solveur de points intérieurs, pour problèmes non linéaires
if not solver.available():
    raise RuntimeError("Ipopt n'est pas trouvé : vérifie son installation et le PATH.")
else:
    print('Ipopt trouvé.')
print("Version :", solver.version())


resultat = solver.solve(model, tee=True)

print("Statut du solveur :", resultat.solver.status)
print("Statut :", resultat.solver.termination_condition)
for v in (model.x1, model.x2, model.x3, model.x4):
    print(f"{v.name} = {pyo.value(v):.4f}")
print(f"Objectif = {pyo.value(model.obj):.4f}")

Ipopt trouvé.
Version : (3, 14, 19, 0)
Ipopt 3.14.19: max_cpu_time=10


******************************************************************************
This program contains Ipopt, a library for large-scale nonlinear optimization.
 Ipopt is released as open source code under the Eclipse Public License (EPL).
         For more information visit https://github.com/coin-or/Ipopt
******************************************************************************

This is Ipopt version 3.14.19, running with linear solver MUMPS 5.8.1.

Number of nonzeros in equality constraint Jacobian...:        4
Number of nonzeros in inequality constraint Jacobian.:        4
Number of nonzeros in Lagrangian Hessian.............:       10

Total number of variables............................:        4
                     variables with only lower bounds:        0
                variables with lower and upper bounds:        4
                     variables with only upper bounds:        0
Total number of equal